# Netflix Customer Churn Analysis

## Objective
The objective of this project is to **predict customer churn in an OTT platform** using machine learning models and to **identify key churn drivers** influencing churn behavior.

## Project Workflow
1. Load and inspect the dataset  
2. Perform data preprocessing and feature preparation  
3. Conduct exploratory data analysis (EDA) to identify churn patterns  
4. Build and evaluate machine learning models  
5. Interpret Logistic Regression coefficients to identify key churn drivers  
6. Compare predictive performance of tree-based models  


## 1. Importing Libraries

In [ ]:

import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix, f1_score, classification_report, roc_curve, roc_auc_score
import joblib


## 2. Loading Dataset

In [ ]:

df = pd.read_csv('netflix_customer_churn.csv')
df.head()


## 3. Data Overview

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

## 4. Data Preprocessing

In [ ]:

df.drop(columns='customer_id', inplace=True)

num_cols = df.select_dtypes(exclude=['object','bool'])
cat_cols = df.select_dtypes(include=['object','bool'])

cat_code = pd.get_dummies(cat_cols, drop_first=True).astype(int)

Not_churn = num_cols.drop(columns='churned')

scaler = StandardScaler()
scaled = scaler.fit_transform(Not_churn)
num_scaled = pd.DataFrame(scaled, columns=Not_churn.columns, index=Not_churn.index)

f_df = pd.concat([num_scaled, cat_code], axis=1)


## 5. Exploratory Data Analysis (EDA)

In [ ]:

sns.countplot(x='churned', data=df)
plt.title("Churn Distribution")
plt.show()


In [ ]:
pd.crosstab(df['subscription_type'], df['churned'])

In [ ]:

sns.barplot(x='churned', y='last_login_days', data=df)
plt.title("Login Days vs Churn")
plt.show()


In [ ]:

sns.barplot(x='payment_method', hue='churned', data=df)
plt.title("Payment Method vs Churn")
plt.show()



### Key Observations (EDA)
- Customers with higher inactivity are more likely to churn  
- Subscription type and payment method influence churn behavior  


## 6. Correlation Analysis

In [ ]:

corr = df.corr(numeric_only=True)
sns.heatmap(corr, cmap='coolwarm', annot=True)
plt.title("Correlation Heatmap")
plt.show()


## 7. Model Building and Evaluation

In [ ]:

X = f_df
y = df['churned']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


### Logistic Regression (Interpretability)

In [ ]:

Log = LogisticRegression()
Log.fit(X_train, y_train)
pred = Log.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print("Recall:", recall_score(y_test, pred))
print("Precision:", precision_score(y_test, pred))
print("F1 Score:", f1_score(y_test, pred))
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred))


### Key Churn Drivers from Logistic Regression

In [ ]:

features = X_train.columns
coeff = Log.coef_[0]

fimport = pd.DataFrame({
    'Feature': features,
    'Coefficient': coeff,
    'Odds_Ratio': np.exp(coeff)
}).sort_values(by='Coefficient', key=abs, ascending=False)

fimport


### ROC Curve

In [ ]:

y_probs = Log.predict_proba(X_test)[:,1]
fpr, tpr, _ = roc_curve(y_test, y_probs)

plt.plot(fpr, tpr, label='ROC Curve')
plt.plot([0,1],[0,1],'--')
plt.legend()
plt.show()

print("ROC AUC:", roc_auc_score(y_test, y_probs))


### Decision Tree Model

In [ ]:

DT = DecisionTreeClassifier(max_depth=5)
DT.fit(X_train, y_train)
dt_pred = DT.predict(X_test)

print("Accuracy:", accuracy_score(y_test, dt_pred))
print("Recall:", recall_score(y_test, dt_pred))
print("Precision:", precision_score(y_test, dt_pred))
print("F1 Score:", f1_score(y_test, dt_pred))
print(confusion_matrix(y_test, dt_pred))
print(classification_report(y_test, dt_pred))


### Random Forest Model

In [ ]:

RF = RandomForestClassifier(n_estimators=200, random_state=42)
RF.fit(X_train, y_train)
rf_pred = RF.predict(X_test)

print("Accuracy:", accuracy_score(y_test, rf_pred))
print("Recall:", recall_score(y_test, rf_pred))
print("Precision:", precision_score(y_test, rf_pred))
print("F1 Score:", f1_score(y_test, rf_pred))
print(confusion_matrix(y_test, rf_pred))
print(classification_report(y_test, rf_pred))


## 8. Conclusion


This notebook demonstrated an end-to-end churn prediction workflow.  
While tree-based models achieved higher predictive accuracy, Logistic Regression was used to interpret model coefficients and identify key churn drivers.  
These insights can support data-driven customer retention strategies in OTT platforms.
